In [ ]:
from collections import namedtuple
import math
import random

State = namedtuple("State", ["pits", "stores", "to_move"])

In [ ]:
class Mancala:
    def __init__(self):
        self.initial = State(
            pits=(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4), 
            stores=(0, 0),
            to_move=0
        )
        self.nodes_explored = 0
    
    def actions(self, state):
        if self.is_terminal(state):
            return []
        
        if state.to_move == 0:
            return [i for i in range(0, 6) if state.pits[i] > 0]
        else:
            return [i for i in range(6, 12) if state.pits[i] > 0]
    
    def result(self, state, action):
        pits = list(state.pits)
        stores = list(state.stores)
        player = state.to_move
        

        stones = pits[action]
        pits[action] = 0
        position = action
        last_landing = None
        

        while stones > 0:
            position = (position + 1) % 12
            

            if position == 6 and player == 0:
                stores[0] += 1
                stones -= 1
                last_landing = 'store0'
                if stones == 0:
                    break
                continue
            

            if position == 0 and player == 1:
                stores[1] += 1
                stones -= 1
                last_landing = 'store1'
                if stones == 0:
                    break
                continue
            

            pits[position] += 1
            stones -= 1
            last_landing = position
        

        if isinstance(last_landing, int):
            own_side = range(6) if player == 0 else range(6, 12)
            if last_landing in own_side and pits[last_landing] == 1:
                opposite = 11 - last_landing
                if pits[opposite] > 0:
                    stores[player] += pits[last_landing] + pits[opposite]
                    pits[last_landing] = 0
                    pits[opposite] = 0
        

        extra_turn = (last_landing == 'store0' and player == 0) or \
                     (last_landing == 'store1' and player == 1)
        next_player = player if extra_turn else 1 - player
        
        new_state = State(tuple(pits), tuple(stores), next_player)
        
        if self.is_terminal(new_state):
            new_state = self._collect_remaining(new_state)
        
        return new_state
    
    def _collect_remaining(self, state):
        pits = list(state.pits)
        stores = list(state.stores)
        
        for i in range(6):
            stores[0] += pits[i]
            pits[i] = 0
        
        for i in range(6, 12):
            stores[1] += pits[i]
            pits[i] = 0
        
        return State(tuple(pits), tuple(stores), state.to_move)
    
    def is_terminal(self, state):
        p0_empty = all(state.pits[i] == 0 for i in range(6))
        p1_empty = all(state.pits[i] == 0 for i in range(6, 12))
        return p0_empty or p1_empty
    
    def utility(self, state, player):
        if self.is_terminal(state):
            final = self._collect_remaining(state)
            return final.stores[player] - final.stores[1 - player]
        return state.stores[player] - state.stores[1 - player]
    
    def display(self, state):
        print()
        print('  P1: ', end='')
        for i in range(11, 5, -1):
            print(f'{state.pits[i]:2}', end=' ')
        print()
        print(f'{state.stores[1]:3}{" "*18}{state.stores[0]:3}')
        print('  P0: ', end='')
        for i in range(6):
            print(f'{state.pits[i]:2}', end=' ')
        print('\n')

In [ ]:
def alphabeta_search(game, state, max_depth=6):
    player = state.to_move
    game.nodes_explored = 0
    
    def max_value(s, alpha, beta, depth):
        game.nodes_explored += 1
        
        if game.is_terminal(s) or depth >= max_depth:
            return game.utility(s, player), None
        
        v, move = -math.inf, None
        for a in game.actions(s):
            v2, _ = min_value(game.result(s, a), alpha, beta, depth + 1)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move
    
    def min_value(s, alpha, beta, depth):
        game.nodes_explored += 1
        
        if game.is_terminal(s) or depth >= max_depth:
            return game.utility(s, player), None
        
        v, move = math.inf, None
        for a in game.actions(s):
            v2, _ = max_value(game.result(s, a), alpha, beta, depth + 1)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move
    
    return max_value(state, -math.inf, math.inf, 0)

In [ ]:
def human_player(game, state):
    print("\nYour turn")
    game.display(state)

    valid_moves = game.actions(state)
    print("Your valid moves:", valid_moves)

    while True:
        try:
            move = int(input("Choose a pit: "))
            if move in valid_moves:
                return move
            else:
                print("Invalid move. Try again.")
        except ValueError:
            print("Please enter a number.")


            
def ai_player(max_depth=6):
    def player(game, state):
        _, move = alphabeta_search(game, state, max_depth)
        return move
    return player

In [ ]:
def play_game(game, players, verbose=False):
    state = game.initial
    
    while not game.is_terminal(state):
        player_num = state.to_move
        move = players[player_num](game, state)
        state = game.result(state, move)
        

    
    return state

In [ ]:
ame = Mancala()


print("human vs AI ")
result = play_game(
    game, 
    {0: human_player, 1: ai_player(max_depth=6)}, 
    verbose=True
)

# Show final result
final = game._collect_remaining(result)
print(f'Final Score: P0={final.stores[0]} P1={final.stores[1]}')

if final.stores[0] > final.stores[1]:
    print('Player 0 wins!')
elif final.stores[1] > final.stores[0]:
    print('Player 1 wins!')
else:
    print('Draw!')